# 📓 Notebook 2: Feature Engineering

Bu notebook'ta:
- URL features çıkaracağız (uzunluk, entropy, karakter oranları)
- Adversarial features ekleyeceğiz (Levenshtein, homoglyph)
- Domain metadata işleyeceğiz (WHOIS, SSL)
- Zero-day zaman bazlı split yapacağız


In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.feature_extractor import FeaturePipeline, time_based_split
from src.utils import load_dataframe, save_dataframe, print_section, FIGURES_DIR

print('✅ Import başarılı')

✅ Import başarılı


In [2]:
# Notebook 1'den gelen veriyi yükle
try:
    df_raw = load_dataframe('raw_dataset')
    print(f'Veri yüklendi: {df_raw.shape}')
except FileNotFoundError:
    print('raw_dataset bulunamadı, önce 1_data_collection.ipynb çalıştırın!')
    raise

df_raw.head()

Veri yüklendi: (50000, 4)


,url,timestamp,label,source
0,kansascity-lifeinsurance.org/,2024-08-02 00:00:00+00:00,0,kaggle
1,http://www.tokabrasil.com.br/~wellsfargo0o/cgi...,2024-06-14 00:00:00+00:00,1,kaggle
2,tinyurl.com/ju7vx9b,2023-03-26 00:00:00+00:00,1,kaggle
3,https://www.kunden-kundenservices.com/vetos/an...,2023-09-22 00:00:00+00:00,1,kaggle
4,youtube.com/watch?v=wkbRmmVGRjk,2024-01-20 00:00:00+00:00,0,kaggle


## ⚙️ Feature Extraction

In [3]:
# Tüm feature'ları çıkar
pipeline = FeaturePipeline()
df_features = pipeline.transform(df_raw)

print(f'\nFeature matrix: {df_features.shape}')
print(f'Feature sayısı: {df_features.shape[1] - 3}')  # url, label, timestamp hariç
df_features.head()


 Feature extraction basliyor...
   [1/4] URL features cikariliyor...


URL Features: 100%|████████████████████████████████████████████████████████████████████| 50000/50000 [00:03<00:00, 12818.10it/s]


   [2/4] Adversarial features cikariliyor...


Adversarial Features: 100%|██████████████████████████████████████████████████████████████| 50000/50000 [00:53<00:00, 937.00it/s]


   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...


Cialdini Features: 100%|███████████████████████████████████████████████████████████████| 50000/50000 [00:03<00:00, 15060.54it/s]


Feature extraction tamamlandi: 50000 satir, 64 feature

Feature matrix: (50000, 67)
Feature sayısı: 64


,url,label,url_length,domain_length,path_length,query_length,subdomain_length,num_subdomains,num_dots,num_hyphens,...,domain_confusion_score,cld_authority,cld_scarcity,cld_fear,cld_reciprocity,cld_social_proof,cld_commitment,cld_total_score,cld_dominant_principle,timestamp
0,kansascity-lifeinsurance.org/,0,29,24,29,0,0,0,1,1,...,0.1,0.0000,0.0,0.0,0.0000,0.0,0.0,0.0000,0,2024-08-02 00:00:00+00:00
1,http://www.tokabrasil.com.br/~wellsfargo0o/cgi...,1,108,10,80,0,3,1,6,2,...,0.1,0.2727,0.0,0.0,0.0000,0.0,0.0,0.0454,0,2024-06-14 00:00:00+00:00
2,tinyurl.com/ju7vx9b,1,19,7,19,0,0,0,1,0,...,0.1,0.0000,0.0,0.0,0.0000,0.0,0.0,0.0000,0,2023-03-26 00:00:00+00:00
3,https://www.kunden-kundenservices.com/vetos/an...,1,237,21,19,180,3,1,3,1,...,0.0,0.0000,0.0,0.0,0.1071,0.0,0.0,0.0179,3,2023-09-22 00:00:00+00:00
4,youtube.com/watch?v=wkbRmmVGRjk,0,31,7,17,13,0,0,1,0,...,0.0,0.7500,0.0,0.0,0.0000,0.0,0.0,0.1250,0,2024-01-20 00:00:00+00:00


In [4]:
# Feature listesi
feature_cols = [c for c in df_features.columns if c not in ['url', 'label', 'timestamp']]
print(f'Toplam {len(feature_cols)} feature:')
for i, f in enumerate(feature_cols):
    print(f'  {i+1:3d}. {f}')

Toplam 64 feature:
    1. url_length
    2. domain_length
    3. path_length
    4. query_length
    5. subdomain_length
    6. num_subdomains
    7. num_dots
    8. num_hyphens
    9. num_underscores
   10. num_slashes
   11. num_question
   12. num_equals
   13. num_at
   14. num_ampersand
   15. num_exclamation
   16. num_tilde
   17. num_percent
   18. num_hash
   19. num_plus
   20. digit_ratio
   21. alpha_ratio
   22. special_ratio
   23. url_entropy
   24. domain_entropy
   25. path_entropy
   26. is_https
   27. has_port
   28. port
   29. tld_suspicious
   30. tld_trusted
   31. tld_length
   32. is_ip
   33. suspicious_keywords
   34. has_login_keyword
   35. has_verify_keyword
   36. has_secure_keyword
   37. has_paypal_keyword
   38. has_account_keyword
   39. path_depth
   40. num_query_params
   41. has_double_slash
   42. domain_has_digit
   43. domain_has_hyphen
   44. long_url
   45. very_long_url
   46. short_url
   47. min_brand_levenshtein
   48. min_brand_levensht

## 📊 Feature Analizi

In [5]:
# Phishing vs Legitimate feature karşılaştırması
key_features = [
    'url_length', 'num_hyphens', 'num_dots', 'url_entropy',
    'domain_entropy', 'digit_ratio', 'num_at', 'suspicious_keywords',
    'min_brand_levenshtein', 'has_homoglyph', 'brand_in_non_brand_domain'
]
available_key = [f for f in key_features if f in df_features.columns]

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, feat in enumerate(available_key[:12]):
    phish  = df_features[df_features['label']==1][feat].dropna()
    legit  = df_features[df_features['label']==0][feat].dropna()
    
    axes[i].hist(legit, bins=30, alpha=0.6, color='#4CAF50', label='Legitimate', density=True)
    axes[i].hist(phish, bins=30, alpha=0.6, color='#F44336', label='Phishing', density=True)
    axes[i].set_title(feat, fontsize=9)
    axes[i].legend(fontsize=7)

for j in range(len(available_key), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Phishing vs Legitimate: Feature Dağılımları', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\emira\AppData\Local\Temp\ipykernel_16952\3851365275.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Korelasyon matrisi (top 20 feature)
top_features = df_features[feature_cols].corrwith(df_features['label']).abs().nlargest(20).index.tolist()

corr_matrix = df_features[top_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, ax=ax,
            annot_kws={'size': 7})
ax.set_title('Feature Korelasyon Matrisi (Top 20 Feature)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\emira\AppData\Local\Temp\ipykernel_16952\1706960172.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Label ile korelasyon
correlations = df_features[feature_cols].corrwith(df_features['label']).abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top_corr = correlations.head(20)
colors = ['#F44336' if v > 0.3 else '#FF9800' if v > 0.1 else '#4CAF50' for v in top_corr.values]
ax.barh(top_corr.index[::-1], top_corr.values[::-1], color=colors[::-1], alpha=0.8)
ax.set_title('Feature-Label Korelasyonu (Top 20)', fontsize=13)
ax.set_xlabel('|Korelasyon|')
ax.axvline(x=0.3, color='red', linestyle='--', alpha=0.5, label='Güçlü (>0.3)')
ax.axvline(x=0.1, color='orange', linestyle='--', alpha=0.5, label='Orta (>0.1)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_label_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nEn yüksek korelasyonlu feature\'lar:')
print(correlations.head(10).to_string())


En yüksek korelasyonlu feature'lar:
is_https                      0.495334
tld_trusted                   0.402913
suspicious_keywords           0.287914
has_login_keyword             0.278639
cld_commitment                0.247634
min_brand_levenshtein         0.232610
is_exact_brand                0.224527
min_brand_levenshtein_norm    0.193893
digit_ratio                   0.178526
num_slashes                   0.176819


C:\Users\emira\AppData\Local\Temp\ipykernel_16952\741832684.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 📅 Zero-Day Split

In [8]:
# Zaman bazlı train/test split
X_train, X_test, y_train, y_test, train_df, test_df, feature_names = time_based_split(
    df_features,
    train_end_year=2023,
    test_start_year=2024
)

print(f'\nX_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'Feature sayısı: {len(feature_names)}')


Zaman Bazli Split:
   TRAIN: 24,987 ornek (<= 2023)
   TEST:  25,013 ornek (>= 2024)
   Feature sayisi: 64

X_train shape: (24987, 64)
X_test shape: (25013, 64)
Feature sayısı: 64


In [9]:
# Split görselleştir
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Train sınıf dağılımı
axes[0].bar(['Legitimate', 'Phishing'],
            [(y_train==0).sum(), (y_train==1).sum()],
            color=['#4CAF50', '#F44336'], alpha=0.8)
axes[0].set_title(f'TRAIN Sınıf Dağılımı\n(n={len(y_train):,})')
axes[0].set_ylabel('Sayı')

# Test sınıf dağılımı
axes[1].bar(['Legitimate', 'Phishing'],
            [(y_test==0).sum(), (y_test==1).sum()],
            color=['#4CAF50', '#F44336'], alpha=0.8)
axes[1].set_title(f'TEST Sınıf Dağılımı (Zero-Day)\n(n={len(y_test):,})')
axes[1].set_ylabel('Sayı')

plt.suptitle('Zaman Bazlı Train/Test Split', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'train_test_split.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\emira\AppData\Local\Temp\ipykernel_16952\1207899380.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# Kaydet
import numpy as np
np.save('../data/processed/X_train.npy', X_train)
np.save('../data/processed/X_test.npy', X_test)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_test.npy', y_test)

import json
with open('../data/processed/feature_names.json', 'w') as f:
    json.dump(feature_names, f)

save_dataframe(df_features, 'feature_matrix')

print('\n✅ Notebook 2 tamamlandı!')
print('   Sonraki adım: 3_model_training.ipynb')

[2026-02-28 12:36:17] INFO [utils] DataFrame kaydedildi: C:\Users\emira\OneDrive\Desktop\main projem\notebooks\..\data\processed\feature_matrix (50000 satır)



✅ Notebook 2 tamamlandı!
   Sonraki adım: 3_model_training.ipynb
